In [21]:
import cohere
import pandas as pd
import os
import ast 
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

load_dotenv()  # Load API key from .env file

OS_HOST = os.getenv("OS_HOST")
OS_API_KEY = os.getenv("OS_API_KEY")

client = Elasticsearch(
    hosts=[OS_HOST],
    api_key=OS_API_KEY
)

# Create the index if it does not exist yet

INDEX_NAME = "vectorhood"

In [22]:
data_sets = [
    # "clothing_items.csv",
    # "fruits_and_vegetables.csv",
    # "household_items.csv",
    # "animals_list.csv",
    # "random_stuff.csv",
    "sentences.csv",
]

In [23]:
# Embed the datasets

COHERE_API_KEY = os.getenv("COHERE_API_KEY")
co = cohere.Client(COHERE_API_KEY)

for data_set in data_sets:
    df = pd.read_csv("data/" + data_set)
    df['text_for_embedding'] = df['Name']

    # Get embeddings from Cohere's english v3 model
    embeddings = co.embed(
        texts=df['text_for_embedding'].tolist(),
        input_type="search_document",
        model="embed-english-light-v3.0"
    ).embeddings

    # Add embeddings to DataFrame
    df['Embedding'] = embeddings

    # Drop the intermediate combined field
    df.drop('text_for_embedding', axis=1, inplace=True)

    # Save the embeddings to a new CSV file
    output_file = os.path.join("out", data_set)
    df.to_csv(output_file, index=False)

    print("✅ Embeddings generated and saved to ", output_file)


✅ Embeddings generated and saved to  out/sentences.csv


In [24]:


if not client.indices.exists(index=INDEX_NAME):
    client.indices.create(index=INDEX_NAME, body={
        "mappings": {
            "properties": {
                "name": {"type": "text"},
                "category": {"type": "text"},
                "embedding": {
                    "type": "dense_vector",
                    "dims": 384,  
                    "index": True,
                    "similarity": "cosine"
                }
            }
        }
    })
    

In [25]:
# query = {
#     "query": {
#         "match": {
#             "category": "Sentence"
#         }
#     }
# }

# # Perform delete by query
# response = client.delete_by_query(index=INDEX_NAME, body=query)

# # print how many documents were deleted
# print(f"Deleted {response['deleted']} documents from the index.")

Bulk ingest the data into the index

In [26]:

for dataset in data_sets:
    # Load your memes CSV
    df = pd.read_csv("out/" + dataset)
    df["Embedding"] = df["Embedding"].apply(ast.literal_eval)

    print(df.head())
    # Build documents for indexing
    documents = [
        {
            "_index": INDEX_NAME,
            "_source": {
                "name": row["Name"],
                "category": row["Category"],
                "embedding": row["Embedding"]
            }
        }
        for i, row in df.iterrows()
    ]

    # Upload using bulk helper
    response = bulk(client, documents)
    print("Bulk upload response:", response)


          Name  Category                                          Embedding
0    kiwi bird  Sentence  [0.000644207, 0.03186035, 0.04336548, 0.054443...
1   kiwi fruit  Sentence  [-0.064575195, -0.00044631958, 0.054016113, 0....
2     kiwi man  Sentence  [-0.049072266, 0.042175293, 0.027786255, 0.064...
3  tuxedo suit  Sentence  [-0.05911255, 0.04815674, 0.016448975, -0.0181...
4   tuxedo cat  Sentence  [-0.059295654, -0.014297485, 0.029815674, -0.0...
Bulk upload response: (9, [])
